# Agent Harness

预计阅读时间：15-20 分钟。

前面几节已经实现了一个逐步增强的 Coding Agent：先让模型调用 Bash，再加入 Skill，最后通过 MCP 接入外部工具。本节不继续写大量新代码，而是按照参考项目的 S01-S20 顺序，把一个完整 Agent Harness 的主要机制串起来。
    **Agent Harness** 可以理解为 Agent 的运行框架。大模型负责推理和决策，Harness 负责把工具、安全、状态、规划、上下文和记忆组织起来，让 Agent 能够持续、可控地完成任务。

参考顺序来自本地 `reference/learn-claude-code`。其中前面已经介绍过的内容会简要回顾，后面的内容作为完整系统的扩展路线。


## S01 Agent Loop：一个循环就够了

最小 Agent 的核心是循环：

```text
用户目标 -> LLM -> 工具调用 -> 执行结果 -> LLM -> ... -> 最终回答
```

模型不直接操作系统，而是请求调用工具。程序执行工具后，把结果作为新的上下文反馈给模型。

前面的 `02_agent.py` 已经实现了这个模式。它说明了 Agent 的核心不是一次性回答，而是“推理、行动、观察、继续推理”的循环。


## S02 Tool Use：多加一个工具

只有 Bash 工具也能完成很多任务，但表达不够清晰。更常见的做法是把高频能力封装成明确工具，例如读取文件、写入文件、运行测试。

工具越具体，模型越容易选择合适动作，Harness 也越容易做权限控制、参数检查和日志记录。


## S03 Permission：执行前做权限判断

Agent 一旦能执行命令，就必须有安全边界。常见检查包括：

- 禁止明显危险命令；
- 限制路径不能逃出工作目录；
- 对删除、覆盖、联网等操作要求确认。

权限检查的目的不是让 Agent 什么都不能做，而是让高风险操作可见、可控。


## S04 Hooks：挂在循环上

当工具越来越多时，如果每个工具内部都写权限检查、日志记录和错误处理，代码会很分散。Hooks 用来把通用逻辑挂到工具调用前后。

常见 Hook 包括：

- `PreToolUse`：执行前检查权限、参数和确认状态；
- `PostToolUse`：执行后记录日志、截断长输出、提取错误。

这样可以保持核心循环稳定，把扩展能力放到循环外侧。


## S05 TodoWrite：没有计划就容易跑偏

复杂任务通常不能一步完成。Todo 工具让 Agent 显式维护计划和进度。

常见状态包括：

- `pending`：尚未开始；
- `in_progress`：正在处理；
- `completed`：已经完成。

Todo 不直接修改代码，但能让模型在执行前规划，在执行中回看任务，减少跑偏。


## S06 Subagent：大任务拆小

Subagent 用来处理局部任务。例如主 Agent 负责整体修复，子 Agent 负责阅读某个模块、查找相关文件或总结日志。

它的核心价值是上下文隔离：

- 主 Agent 保持主线；
- 子 Agent 拿到干净上下文；
- 子 Agent 完成后只返回摘要或结果。

这可以减少主上下文被中间探索过程污染。


## S07 Skill Loading：用到时才加载

Skill 是可复用的任务经验，通常保存为 `SKILL.md`。它包含任务说明、约束、步骤和示例。

典型流程是：

1. 启动时只加载 Skill 名称和简介；
2. 模型判断任务匹配某个 Skill；
3. 调用 `load_skill` 读取完整内容；
4. 将 Skill 内容作为工具结果返回模型。

前面的 `03_agent.py` 已经实现了这个机制。


## S08 Context Compact：上下文总会满

Agent 长时间工作后，上下文会变长。压缩机制用于控制上下文大小，同时尽量保留重要信息。

常见层次包括：

- 先裁剪无关旧消息；
- 再压缩长工具结果；
- 必要时让 LLM 对历史做摘要；
- 出现上下文超限时执行应急压缩。

核心原则是：便宜的压缩先做，昂贵的 LLM 摘要后做。


## S09 Memory：压缩会丢细节

压缩解决的是当前会话太长的问题，但它可能丢掉长期偏好和项目约定。Memory 用于跨任务保留重要信息。

常见记忆可以分为：

| 类型 | 含义 | 示例 |
| --- | --- | --- |
| user memory | 用户长期偏好 | 默认中文、回答简洁 |
| feedback memory | 反馈形成的规则 | 不要使用绝对路径 |
| project memory | 项目约定 | notebook 控制阅读时间 |
| reference memory | 外部资料索引 | 官方文档、论文链接 |

记忆不应该全部塞进 prompt，而应该按任务检索相关内容。


## S10 System Prompt：运行时组装

早期示例常把 system prompt 写死在代码里。真实 Agent 往往需要根据运行状态动态组装 prompt。

例如：

- 当前工作目录；
- 当前可用工具；
- 已连接的 MCP server；
- 检索到的相关记忆；
- 当前任务状态。

运行时组装能让 prompt 反映真实状态，也方便做缓存和分段管理。


## S11 Error Recovery：错误不是结束

工具执行失败不是终点，而是新的观察结果。

常见错误包括：

- 命令失败；
- 文件不存在；
- 测试不通过；
- 输出格式不符合预期；
- API 调用失败。

Harness 需要把错误反馈给模型，并控制重试次数、退避策略和失败升级方式。
以API调用失败为例，如果是超限可以升级max_tokens，压缩context；如果是模型不可用，则可以选择在随机时间后再次调用。


## S12 Task System：目标拆成小任务

Todo 更偏向当前执行计划，Task System 更偏向持久化任务管理。
1. 拆解任务（Task Decomposition）
2. 管理任务状态（Task State）
3. 处理任务依赖（Dependency Graph）

一个 Task 通常包含：

- 任务 ID；
- 任务描述；
- 状态；
- 依赖关系；
- 负责人或执行者。

任务系统让大目标可以被拆分、排序、恢复和交接。最小状态机可以先理解为三个状态、两个动作：

```text
pending ──claim──→ in_progress ──complete──→ completed
```

其中 `claim` 表示认领任务，`complete` 表示完成任务。


## S13 Background Tasks：慢操作放后台

有些操作耗时很长，例如跑完整测试、训练模型、批量处理文件。后台任务可以让 Agent 不必一直阻塞等待。

典型做法是：

1. 启动后台任务，保存后台任务状态，返回任务 ID；
2. 主 Agent 继续处理其他事情；
3. 后续查询任务状态和结果。

这对环境安装、深度学习模型训练尤其常见。


## S14 Cron Scheduler：按时间表生产工作

Cron Scheduler 让 Agent 可以按时间触发任务。

例如：

- 每天检查一次实验日志；
- 每小时同步一次数据；
- 定期扫描未完成任务；
- 定时生成报告。

它把“用户手动发起任务”扩展为“系统按计划产生任务”。参考实现采用 queue 策略，将调度和执行解耦：

1. Scheduler 线程定期检查 cron 表达式是否到点；
2. 到点后，把任务放入 `cron_queue`；
3. Queue Processor 发现队列有任务，并且 Agent 空闲时，启动一轮执行；
4. Agent Loop 从队列取出任务，把它作为用户消息注入上下文。

这种设计的好处是：调度器只负责“什么时候产生任务”，Agent 只负责“如何执行任务”，两者不会互相阻塞。


## S15 Agent Teams：一个搞不定就组队

当任务很复杂时，可以由多个 Agent 分工协作。例如 Lead 负责拆任务和汇总，队友分别处理后端、前端、测试或文档。

参考实现用“信箱”来解释 Agent Team：每个 Agent 都有一个自己的收件箱，消息用 JSONL 文件保存。

```text
Lead ──send──→ teammate inbox
Lead ←─result── teammate inbox
```

基本机制包括：

- `MessageBus`：消息总线，本质上是文件信箱；
- `.mailboxes/{agent}.jsonl`：每个 Agent 的收件箱；
- `spawn_teammate_thread`：启动一个队友线程；
- `send_message`：向某个 Agent 的信箱追加一条消息；
- `check_inbox`：读取 Lead 的信箱；
- inbox injection：把队友发来的消息注入 Lead 的 `history`，让 Lead 的 LLM 能看到。

这种设计的关键点是：队友有自己的上下文和工具，可以独立工作；协作信息通过信箱传递，而不是共享同一个长上下文。

实际的多Agent系统通常采用对等通信机制。


## S16 Team Protocols：队友之间要有约定

多 Agent 协作不能只靠自由聊天。有些消息只是普通沟通，例如补充要求、同步进展、提交结果；但有些消息属于控制流，例如请求某个队友优雅退出，或者要求队友先提交计划、等待审批后再继续。这类控制动作更适合用 Request/Response 的结构化协议处理，而不是混在普通对话里。

协议通常规定：

- 消息类型；
- 请求 ID；
- 谁发给谁；
- 需要什么格式的回复；
- 是否需要审批。

在参考实现里，协议消息会带上 `type` 和 `metadata.request_id`。发送方先把请求登记到 `pending_requests`，再把消息投递到对方 inbox；接收方看到特定类型的协议消息后，走专门的处理逻辑，并用同一个请求 ID 回一条 response；Lead 之后再从 inbox 里读取响应，把它匹配回原请求，并把状态更新为 approved 或 rejected。

因此，这里的关键不是“所有消息都结构化”，而是“控制类协作结构化，普通协作继续自然对话”。这样既保留了 Agent 之间交流的灵活性，又让关键流程具备可追踪、可恢复、可调试的状态管理。


## S17 Autonomous Agents：自己看板，自己认领

自治 Agent 不只是等待用户输入，而是会在空闲时主动检查任务看板。

基本机制包括：

- 空闲轮询；
- 扫描未认领任务；
- 检查依赖是否完成；
- 自动认领可执行任务。

这让 Agent 从“被动问答”变成“主动处理待办”。


## S18 Worktree Isolation：各干各的

多个任务或多个 Agent 同时修改同一个仓库时，容易互相干扰。Git worktree 可以为不同任务创建独立工作目录。

这样做的好处是：

- 每个任务有自己的目录；
- 每个目录对应自己的分支；
- 实验和修复互不覆盖；
- 最后再统一审查和合并。

这对并行实验和多 Agent 协作很有用。


## S19 MCP Tools：外接工具，标准协议

MCP（Model Context Protocol）定义了 Agent 如何发现和调用外部工具。

前面的 `04_agent.py` 用 mock server 演示了最小流程：

1. 调用 `connect_mcp(name)` 连接外部服务；
2. server 返回可用工具列表；
3. Harness 把工具加入工具池；
4. 模型后续调用 `mcp__server__tool` 形式的工具。

MCP 解决的是“外部工具如何标准化接入”的问题。


## S20 Comprehensive Agent：全部机制归到一个循环

最后，所有机制都要回到同一个核心循环：

```text
读取状态 -> 组装 prompt -> 调用模型 -> 执行工具 -> 更新状态 -> 继续循环
```

完整 Agent Harness 会把工具、安全、Hooks、Todo、SubAgent、Skill、压缩、记忆、任务系统、后台任务、团队协议、worktree 和 MCP 组织在一起。

看起来机制很多，但它们都服务于同一个目标：让模型能够在真实工程环境中持续、可控、可恢复地完成任务。


## 小结

本节按照参考项目 S01-S20 的顺序，对 Agent Harness 做了完整概览。

前面课程已经实现和演示过其中几项：

- `02_agent.py`：Agent Loop + Bash Tool；
- `03_agent.py`：Skill Loading；
- `04_agent.py`：MCP Tools。

后续如果继续扩展，可以从 Todo、Context Compact、Memory 和 System Prompt 这几层开始，因为它们最容易在单 Agent 场景中体现价值，也最贴近日常 Coding Agent 的使用体验。
